In [42]:
import numpy as np 
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,StandardScaler
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [43]:
train = pd.read_csv('train_galactic_wars.csv')
test = pd.read_csv('test_galactic_wars.csv')

print(train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   FightID               4999 non-null   int64  
 1   weapon_jedai          4999 non-null   object 
 2   armour_jedai          4999 non-null   object 
 3   weapon_calmtrooper    4999 non-null   object 
 4   armour_calmtrooper    4999 non-null   object 
 5   injuries              4999 non-null   object 
 6   number_of_fights      4999 non-null   float64
 7   force_level           4999 non-null   float64
 8   accuracy_calmtrooper  4999 non-null   float64
 9   fight_planet          4999 non-null   object 
 10  weather_conditions    4999 non-null   object 
 11  surprise_attack       4999 non-null   bool   
 12  moral_jedai           4999 non-null   float64
 13  moral_calmtrooper     4999 non-null   float64
 14  winner                4999 non-null   int64  
dtypes: bool(1), float64(5

In [44]:
s1_answer = test[(test['fight_planet'] == 'Nabira') & (test['weather_conditions'] == 'Snow')]
print(s1_answer) 

      FightID         weapon_jedai armour_jedai weapon_calmtrooper  \
15       1191    Single Lightsaber        47.0%   Standard Blaster   
105      4021    Double Lightsaber        85.0%     Double Blaster   
117      5624    Double Lightsaber        89.7%     Double Blaster   
124       446    Single Lightsaber        44.2%   Standard Blaster   
128      5346    Double Lightsaber        78.9%     Double Blaster   
...       ...                  ...          ...                ...   
3892     3635    Double Lightsaber        86.5%     Double Blaster   
3893     3363    Double Lightsaber        82.5%     Double Blaster   
3919     7741  Ancient Jedai Blade        58.6%      Heavy Blaster   
3974     3834    Double Lightsaber        90.6%     Double Blaster   
3981     6720  Ancient Jedai Blade        55.9%      Heavy Blaster   

     armour_calmtrooper     injuries  number_of_fights  force_level  \
15                84.1%        Jedai              78.8         30.6   
105              

In [45]:
s2_answer = (train['weapon_calmtrooper'] == 'Experimental Weapon').sum()

train['weapon_calmtrooper'] = train['weapon_calmtrooper'].replace('Experimental Weapon', 'Double Blasten')

print(s2_answer)

1646


In [46]:
#Subtask 3 - clustering

#transform percent columns into num columns
perc_cols = ['armour_jedai', 'armour_calmtrooper']

for col in perc_cols:
    train[col] = train[col].str.rstrip('%').astype(float)

for col in perc_cols: 
    test[col] = test[col].str.rstrip('%').astype(float)
#transform bool columns into num columns
train['surprise_attack'] = train['surprise_attack'].astype(int)
test['surprise_attack'] = test['surprise_attack'].astype(int)

num_cols = ['number_of_fights','force_level','accuracy_calmtrooper','moral_jedai','moral_calmtrooper','armour_jedai', 'armour_calmtrooper','surprise_attack']
cat_cols = ['weapon_jedai','weapon_calmtrooper','injuries','fight_planet','weather_conditions']
y = train['winner']

X = train[num_cols+cat_cols]
X_test = test[num_cols+cat_cols]

num_pipeline = Pipeline(steps = [
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers = [
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

kmeans = KMeans(n_clusters = 3, init = 'k-means++', n_init = 50, random_state = 42)

model_s3 = Pipeline(steps = [
    ('preprocess', preprocessor),
    ('clustering', kmeans)
])

model_s3.fit(X)
clusters_train = model_s3.predict(X)
clusters_test = model_s3.predict(X_test)

print(clusters_test)


[0 1 0 ... 1 1 1]


In [47]:
#Subtask 4

X_tr,X_val,y_tr,y_val = train_test_split(X,y,random_state = 42, test_size = 0.2,stratify = y)

rf = RandomForestClassifier(
    n_estimators = 1000,
    max_depth = 7,
    random_state = 42,
    min_samples_leaf = 3,
    n_jobs = -1,
)

model_s4 = Pipeline(steps = [
    ('preprocess', preprocessor),
    ('clf', rf)
])

model_s4.fit(X_tr,y_tr)

y_pred = model_s4.predict(X_val)

f1 = f1_score(y_val,y_pred,average = 'macro')
print(f'The f1 score for the rf clsf model is {f1}')

model_s4.fit(X,y)

y_test = model_s4.predict(X_test)

The f1 score for the rf clsf model is 0.9683342271253097


In [48]:
submission = pd.DataFrame({
    'subtaskID': np.concatenate([
        [1],
        [2],
        [3] * len(test['FightID']),
        [4] * len(test['FightID'])
    ]),
    'datapointID': np.concatenate([
        [1],
        [2],
        test['FightID'],
        test['FightID']
    ]),
    'answer' : np.concatenate([
        [len(s1_answer)],
        [s2_answer],
        clusters_test,
        y_test
    ])
})

submission.to_csv('submission_3_mar7th.csv')

print(f'This is the submission 3: {submission}')

This is the submission 3:       subtaskID  datapointID  answer
0             1            1     143
1             2            2    1646
2             3         1596       0
3             3         4432       1
4             3          486       0
...         ...          ...     ...
7999          4         4843       1
8000          4         6117       0
8001          4         5810       1
8002          4         3589       1
8003          4         4163       1

[8004 rows x 3 columns]
